In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor
import matplotlib.colors as mcolors

sns.set_style("white")
sns.set(font_scale = 1.5)
sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)
plt.rcParams['axes.formatter.limits'] = (-99, 99)
palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

gobp = pd.read_csv("../DPSC_CNCC/GOBP_AccelEvol_Input.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("../DPSC_CNCC/HPO_AccelEvol_Input.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")

def add_moving_average(df, window_size, column="PhyloP Non", output_col=None):
    """
    Adds a moving average column to a presorted DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Presorted DataFrame
    window_size : int
        Size of the moving window
    column : str
        Column to compute the moving average on
    output_col : str or None
        Name of the output column (defaults to f"{column}_MA_{window_size}")

    Returns
    -------
    pandas.DataFrame
        DataFrame with moving average column added
    """
    if output_col is None:
        output_col = f"{column}_MA_{window_size}"

    df[output_col] = (
        df[column]
        .rolling(window=window_size, min_periods=1, center=True)
        .mean()
    )

    return df

def plot_ma(v, width = 2):
    x = v["Pos"]  # or df["position"] if you have one
    y = v["PhyloP Non MA"]
    
    # Choose a base color
    line_color = "grey"
    
    # Convert to RGBA and make a lighter version
    r, g, b, _ = mcolors.to_rgba(line_color)
    line_color = (r, g, b, 0.7)
    fill_color = (r, g, b, 0.2)  # lower alpha = lighter fill
        
    # Plot smoothed line
    plt.plot(x, y, color=line_color, linewidth=width, zorder = 2)
    
    # Fill between 0 and the line
    plt.fill_between(x, 0, y, color=fill_color, zorder = 1)
    
    #plt.axhline(0, color="black", linewidth=0.8)
    #plt.xlabel("Index")
    #plt.ylabel("Smoothed Moving Average")

import matplotlib.colors as mcolors

def windowed_phyloP_scores(
    zp: pd.DataFrame,
    start: int,
    end: int,
    size: int = 500,
    chrom: str | None = None,
    pos_col: str = "Pos",
    value_col: str = "PhyloP Non",
    category_col: str = "Derived",
    categories: tuple = ("Human", "Chimp"),
):
    """
    Compute windowed sums of PhyloP scores for arbitrary window sizes.

    Parameters
    ----------
    zp : pd.DataFrame
        Input dataframe containing positions, scores, and categories
    start : int
        Start coordinate (inclusive)
    end : int
        End coordinate (exclusive)
    size : int, default 500
        Window size in base pairs
    chrom : str or None
        Chromosome label to attach to output
    pos_col : str
        Column with genomic positions
    value_col : str
        Column to sum within windows
    category_col : str
        Column defining categories (e.g. Human / Chimp)
    categories : tuple
        Categories to extract and difference

    Returns
    -------
    pd.DataFrame
        Windowed summary with one row per window
    """

    df = zp.copy()

    # Ensure integer positions
    df[pos_col] = df[pos_col].astype(int)

    # Restrict to requested region
    df = df[(df[pos_col] >= start) & (df[pos_col] < end)]

    # Assign windows
    df["window_start"] = ((df[pos_col] - start) // size) * size + start
    df["window_end"] = df["window_start"] + size

    # Aggregate
    grouped = (
        df.groupby(
            ["window_start", "window_end", category_col],
            as_index=False
        )[value_col]
        .sum()
    )

    # Pivot to wide format
    wide = (
        grouped
        .pivot_table(
            index=["window_start", "window_end"],
            columns=category_col,
            values=value_col,
            fill_value=0
        )
        .reset_index()
    )

    # Ensure requested categories exist
    for cat in categories:
        if cat not in wide:
            wide[cat] = 0

    # Difference (first - second)
    if len(categories) == 2:
        wide["diff"] = wide[categories[0]] - wide[categories[1]]

    # Add chromosome column if provided
    if chrom is not None:
        wide.insert(0, "chrom", chrom)

    return wide

def make_perm(zp, perm_phylo = False, n_iters = 100, window = 3):
    zpp = zp.copy()
    for i in range(n_iters):
        if perm_phylo:
            zpp["PhyloP Non"] = list(zp.sample(frac = 1, replace = False)["PhyloP Non"])
        zpp["Derived"] = np.random.choice(a=["Human", "Chimp"], p=[0.5, 0.5], size = zpp.shape[0])
        dfp = windowed_phyloP_scores(zpp, start= np.min(zpp["Pos"]), end = np.max(zpp["Pos"]), size = 500)
        dfp.index = dfp["window_end"]
        dfp = dfp[["diff"]]
        dfpma = add_moving_average(dfp, window, column="diff", output_col="PhyloP Non MA " + str(i))
        dfp = dfp[["PhyloP Non MA " + str(i)]]
        
        if i:
            dfpa = dfpa.join(dfp)
        else:
            dfpa = dfp.copy()
    dfpa["Mean"] = np.mean(dfpa, axis = 1)
    dfpa["STD"] = np.std(dfpa, axis = 1)
    dfpa = dfpa[["Mean", "STD"]]
    dfpa["Lower"] = dfpa["Mean"] - dfpa["STD"]
    dfpa["Upper"] = dfpa["Mean"] + dfpa["STD"]
    return dfpa

def insert_zero_crossings(x, y):
    x = list(x)
    y = list(y)
    x_new = []
    y_new = []

    for i in range(len(y) - 1):
        x_new.append(x[i])
        y_new.append(y[i])

        # Sign change (exclude exact zeros)
        if y[i] * y[i + 1] < 0:
            # Linear interpolation for y=0
            t = -y[i] / (y[i + 1] - y[i])
            x0 = x[i] + t * (x[i + 1] - x[i])

            x_new.append(x0)
            y_new.append(0.0)

    x_new.append(x[-1])
    y_new.append(y[-1])

    return np.array(x_new), np.array(y_new)
    
def plot_ma_nc(v, width = 2, fill = True):
    
    # Choose a base color
    hline_color = palette["Human"]
    cline_color = palette["Chimp"]
    
    x = v["Pos"]  # or df["position"] if you have one
    y = v["PhyloP Non MA"].to_numpy()

    x, y = insert_zero_crossings(x, y)
    
    y_pos = np.where(y >= 0, y, np.nan)
    y_neg = np.where(y <= 0, y, np.nan)

    # Convert to RGBA and make a lighter version
    r, g, b, _ = mcolors.to_rgba(hline_color)
    hline_color = (r, g, b, 1)
    hfill_color = (r, g, b, 0.5)  # lower alpha = lighter fill

    r, g, b, _ = mcolors.to_rgba(cline_color)
    cline_color = (r, g, b, 1)
    cfill_color = (r, g, b, 0.5)  # lower alpha = lighter fill
    
    plt.plot(x, y_pos, color=hline_color, linewidth=2)
    plt.plot(x, y_neg, color=cline_color, linewidth=2)

    if fill:
        # Fill between 0 and the line
        plt.fill_between(x, 0, y, color=hfill_color, where=(y > 0), zorder = 1)
        plt.fill_between(x, 0, y, color=cfill_color, where=(y < 0), zorder = 1)
    
    #plt.axhline(0, color="black", linewidth=0.8)
    #plt.xlabel("Index")
    #plt.ylabel("Smoothed Moving Average")



In [ ]:
z = pd.read_csv("HumChp_All_PhyloP.txt.gz", sep = "\t")

In [ ]:
z3 = z[(z["Category"] == "3UTR") & (z["KeptAfterFilt"] == "Y") & (z["SpecSup447"] > 250)]

In [ ]:
v = pd.read_csv("RIMKLB_3UTR_PhyloP447.bed", sep = "\t", header = None)
v = v[[0, 5, 6, 10]]
v.columns = ["Chrom", "Pos", "PhyloP", "Species support"]
v["PhyloP"] = v["PhyloP"].replace(".", 0)
v["PhyloP"] = v["PhyloP"].astype(float)
v["Species support"] = v["Species support"].replace(".", 0)
v["Species support"] = v["Species support"].astype(float)


In [ ]:
v = pd.read_csv("Supplemental_Tables/Supplemental_Table9.csv")
for i in v.columns:
    if not i.startswith("Posi") and not i == "PhyloP" and not i.startswith("Number"):
        v[i] = -v[i]
v

In [ ]:
#Plotting accelerated evolution for RIMKLB
gene = "RIMKLB"
z3p = z3[z3["NearestGene"] == "RIMKLB"]
z3p["Pos"] = [int(x.split(":")[1]) for x in z3p["Position"]]

hum_vars = [int(x.split(":")[1]) for x in z3p[z3p["Derived"] == "H"]["Position"]]
chp_vars = [int(x.split(":")[1]) for x in z3p[z3p["Derived"] == "C"]["Position"]]

region_plot = pd.DataFrame([["chr12", 8773319 - 50, 8773319, "Intron"], ["chr12", 8773319, 8773781, "CDS"], ["chr12", 8773781, 8777190, "UTR"]])
prev = np.min(region_plot[1])
fig, ax = plt.subplots(figsize=(6, 4), dpi=600)
plt.xlim(np.min(region_plot[1]), np.max(region_plot[2]) + 50)

v["PhyloP Non"] = np.maximum(0, v["PhyloP"])
v = add_moving_average(v, 20, output_col = "PhyloP Non MA")
plot_ma(v)

v2 = v[v["Pos"].isin(hum_vars + chp_vars)]
v2 = v2.sort_values("Pos").reset_index(drop=True)

import matplotlib.transforms as transforms

PIXEL_OFFSET = 0.4  # points
MAX_LANES = 5     # limit spread

prev = 0
for i, (_, row) in enumerate(v2.iterrows()):
    
    lane = i % MAX_LANES
    dx = (lane - (MAX_LANES - 1) / 2) * PIXEL_OFFSET

    trans = transforms.offset_copy(
        ax.transData,
        fig=fig,
        x=dx,
        y=0,
        units="points"
    )

    x = np.linspace(row["Pos"] - 0.5, row["Pos"], 10)

    if row["Pos"] in hum_vars:
        color = palette["Human"]
    elif row["Pos"] in chp_vars:
        color = palette["Chimp"]
    else:
        continue
    if row["Pos"] - prev < 10:
        print("BLEH")
        ax.fill_between(
            x,
            0,
            max(row["PhyloP"], 0),
            color=color,
            transform=trans,
            zorder=3
        )
    else:
        ax.fill_between(
            x,
            0,
            max(row["PhyloP"], 0),
            color=color,
            zorder=3
        )
    prev = row["Pos"]

for index, row in region_plot.iterrows():
    if row[3] == "CDS":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.3, y2=-0.3, color='black', zorder = 4)
    if row[3] == "UTR":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.13, y2=-0.13, color='black', zorder = 4)
    if row[3] == "Intron":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.075, y2=-0.075, color='black', zorder = 4)
        prev = row[2]
    elif prev == np.min(region_plot[1]):
        pass
    #plt.axis('off')
plt.title("Human accelerated evolution of $\it{RIMKLB}$ 3' UTR", size = 16)
plt.xticks([8773500, 8775500, 8777000], ["chr12:8773500", "chr12:8775500", "chr12:8777000"], size = 14)
plt.xlabel("Genomic position", size = 16)
plt.ylabel("PhyloP")

In [ ]:
#Plotting for nonsynonymous for RET and other genes
z = pd.read_csv("HumChp_Mis_Final_Tri447.bed", sep = "\t", header = None)
z2 = pd.read_csv("HumChp_Mis_Final_ChpDer_Tri447.bed", sep = "\t", header = None)
z["Derived"] = np.repeat("H", z.shape[0])
z2["Derived"] = np.repeat("C", z2.shape[0])
z = pd.concat([z, z2])
z["Position"] = z[0] + ":" + z[2].astype(str)
z = z[["Position", 3, 5, 11, "Derived"]]
z = z[z[5] != "."]
z[5] = z[5].astype(float)
z.index = z["Position"]
z.columns = ["Position", "Gene", "PhyloP", "Tri", "Derived"]
z["PhyloP Non"] = np.maximum(0, z["PhyloP"])
z

In [ ]:
gene = "ESR1"
b = pd.read_csv("CDS_PhyloP447_Output.bed", sep = "\t", header = None)
bg = b[b[3] == gene + "_CDS"]
bg = bg.sort_values(6)
bg["Posi"] = bg[6]
bg["Pos"] = list(range(1, bg.shape[0]+1))
bg["PhyloP Non"] = np.maximum(bg[7], 0)
zp = z[z["Gene"] == gene]
bgp = bg[bg["Pos"] % 3 != 0]
np.mean(bgp["PhyloP Non"])

In [ ]:
bgp = add_moving_average(bgp, 20, output_col = "PhyloP Non MA")

bgp

In [ ]:


hum_vars = [int(x.split(":")[1]) for x in zp[zp["Derived"] == "H"]["Position"]]
chp_vars = [int(x.split(":")[1]) for x in zp[zp["Derived"] == "C"]["Position"]]

region_plot = pd.DataFrame([["$\it{" + gene + "}$ CDS", np.min(bg["Pos"]), np.max(bg["Pos"]), "CDS"]])
prev = np.min(region_plot[1])
plt.figure(figsize=(6, 4), dpi=600)
plt.xlim(np.min(region_plot[1]) - 50, np.max(region_plot[2]) + 50)

c = 0

bg = bg.sort_values("Pos")
bgp = add_moving_average(bgp, 5, output_col = "PhyloP Non MA")
plot_ma(bgp, width = 0.8)

for index, row in bg.iterrows():
    if row["Posi"] in hum_vars:
        plt.fill_between(np.linspace(row["Pos"] - 1, row["Pos"] + 0, 10), y1=max(row[7], 0), y2=0, color=palette["Human"], zorder = 3)
        c += 1
    elif row["Posi"] in chp_vars:
        plt.fill_between(np.linspace(row["Pos"] - 1, row["Pos"] + 0, 10), y1=max(row[7], 0), y2=0, color=palette["Chimp"], zorder = 3)
        c += 1


for index, row in region_plot.iterrows():
    if row[3] == "CDS":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.3, y2=-0.3, color='black', zorder = 4)
    if row[3] == "UTR":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.13, y2=-0.13, color='black', zorder = 4)
    if row[3] == "Intron":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.075, y2=-0.075, color='black', zorder = 4)
        prev = row[2]
    elif prev == np.min(region_plot[1]):
        pass
    #plt.axis('off')
plt.title("Human accelerated evolution of $\it{" + gene + "}$ CDS", size = 16)
plt.xticks([0, 303*3, np.max(bg["Pos"])], [0, 303, np.max(bg["Pos"])//3], size = 14)
plt.xlim
plt.xlabel("Position in $\it{" + gene + "}$ CDS", size = 16)
plt.ylabel("PhyloP", size = 14)

In [ ]:
s = pd.read_csv("Supplemental_Tables/Supplemental_Table5.csv")
keep = list(s[s["Acceleration"] == "Human-accelerated, FDR < 0.05"]["Gene symbol"])

In [ ]:
#For non-coding
z = pd.read_csv("HumChp_All_PhyloP.txt.gz", sep = "\t")
z = z[(z["SpecSup447"] > 250) & (z["Category"] == "NC")]
z["PhyloP Non"] = np.maximum(0, z["PhyloP447"])
z = z.drop_duplicates("Position")
z["Derived"] = z["Derived"].replace("H", "Human").replace("C", "Chimp")
z

In [ ]:
#Used this to do the nonsyn and syn versions
z = pd.read_csv("HumChp_All_PhyloP.txt.gz", sep = "\t")
zx = z[z["Category"].isin(["_Syn", "NC_Syn"])]
zy = z[z["Category"].isin(["Mis"])]
print(zx[(zx["PhyloP447"] > 3) & (zx["Derived"] == "C")].shape)
print(zx[(zx["Derived"] == "C") & (zx["KeptAfterFilt"] == "Y")].shape)

In [ ]:
#All these were computed as above
fisher_exact([[10918, 9642], [43579, 44569]])

In [ ]:
fisher_exact([[10918, 9642], [4218, 4113]])

In [ ]:
fisher_exact([[38856, 41442], [43579, 44569]])

In [ ]:
fisher_exact([[38856, 41442], [39361, 40453]])

In [ ]:
fisher_exact([[38856 + 10918, 41442 + 9642], [43579, 44569]])

In [ ]:
p1 = pd.read_csv("AccelEvol_Pseudos.bed", sep = "\t", header = None, usecols = [0, 1, 2])

pseudo = list(np.unique(p1[0] + ":" + p1[2].astype(str)))
z = z[~z["Position"].isin(pseudo)]
z

In [ ]:
Counter(zx["Derived"])

In [ ]:
#Creating Supp. Table 7
x = pd.DataFrame()
for gene in keep:
    zp = z[z["NearestGene"] == gene]
    zp = zp[zp["KeptAfterFilt"] == "Y"]
    zp["Pos"] = [int(x.split(":")[1]) for x in zp["Position"]]
    zp = zp.sort_values("Pos")
    chrom = list(zp["Position"])[0].split(":")[0]
    start = list(zp["Pos"])[0]
    end = list(zp["Pos"])[zp.shape[0]-1]
    out = []
    size = 500
    df = windowed_phyloP_scores(zp, start= np.min(zp["Pos"]), end = np.max(zp["Pos"]), size = size)
    df["Gene symbol"] = np.repeat(gene, df.shape[0])
    df["Chrom"] = np.repeat(chrom, df.shape[0])
    x = pd.concat([x, df])

x = x.sort_values("diff")
x["Genomic region, hg38"] = x["Chrom"] + ":" + x["window_start"].astype(str) + "-" + x["window_end"].astype(str)
x = x[["Genomic region, hg38", "Human", "Chimp", "diff", "Gene symbol"]]
x.columns = ["Genomic region, hg38", "Human PhyloP-weighted sum", "Chimp PhyloP-weighted sum", "Difference in PhyloP-weighted sum", "Gene symbol"]
x = x.sort_values("Difference in PhyloP-weighted sum", ascending = False)
x.to_csv("Supplemental_Tables/Supplemental_Table7.csv")

In [ ]:
gene = "SIX3"
zp = z[z["NearestGene"] == gene]
zp = zp[zp["KeptAfterFilt"] == "Y"]
zp["Pos"] = [int(x.split(":")[1]) for x in zp["Position"]]
zp = zp.sort_values("Pos")
chrom = list(zp["Position"])[0].split(":")[0]
start = list(zp["Pos"])[0]
end = list(zp["Pos"])[zp.shape[0]-1]
out = []
size = 500
df = windowed_phyloP_scores(zp, start= np.min(zp["Pos"]), end = np.max(zp["Pos"]), size = size)


In [ ]:
ma_window_size = 10

dfma = add_moving_average(df, ma_window_size, column="diff", output_col="PhyloP Non MA")
dfma

In [ ]:
dfma = add_moving_average(df, ma_window_size, column="diff", output_col="PhyloP Non MA")
dfma["window_start"] = list(dfma["window_start"])[::-1]
dfma["window_end"] = list(dfma["window_start"])[::-1]
dfma["Pos"] = dfma["window_end"]
dfma = dfma.sort_values("Pos")
dfma

In [ ]:
c = 0
ma_window_size = 10
fig, ax = plt.subplots(figsize=(6, 4), dpi=600)
dfpa = make_perm(zp, window = ma_window_size)

r, g, b, _ = mcolors.to_rgba("grey")
fill_color = (r, g, b, 0.2)
plt.plot(list(dfpa.index), dfpa["Lower"], color="black", zorder = 3, linewidth = 0.8)
plt.plot(list(dfpa.index), dfpa["Upper"], color="black", zorder = 3, linewidth = 0.8)

dfma = add_moving_average(df, ma_window_size, column="diff", output_col="PhyloP Non MA")
dfma["window_start"] = list(dfma["window_start"])[::-1]
dfma["window_end"] = list(dfma["window_start"])[::-1]
dfma["Pos"] = dfma["window_end"]
dfma = dfma.sort_values("Pos")
plot_ma_nc(dfma, width = 0.5, fill = True)

regions_file = pd.read_csv("All_NC_Regions.bed", sep = "\t", header = None)
regions_file = regions_file[(regions_file[3].isin(["CDS", "UTR"])) & (regions_file[4] == gene)].sort_values(2)

prev_ind = 0
prev = 0
out = []
for index, row in regions_file.iterrows():
    if prev_ind:
        if prev[2] == row[1]:
            out.append(list(row))
        else:
            out.append([row[0], prev[2], row[1], "Intron"])
            out.append(list(row))
    else:
        out.append(list(row))
        prev_ind = 1
    prev = row
region_plot = pd.DataFrame(out)

adjust = (np.max(dfma["PhyloP Non MA"]) - np.min(dfma["PhyloP Non MA"]))/10
for index, row in region_plot.iterrows():
    #if row[3] == "CDS":
    #    plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.3*adjust, y2=-0.3*adjust, color='black', zorder = 4)
    #if row[3] == "UTR":
    #    plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.13*adjust, y2=-0.13*adjust, color='black', zorder = 4)
    #if row[3] == "Intron":
    #    plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.075*adjust/2, y2=-0.075*adjust/2, color='black', zorder = 4)
    if row[3]:
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.2*adjust, y2=-0.2*adjust, color='black', zorder = 4)
plt.ylabel("Difference in PhyloP moving average", size = 13)
plt.title("Human accelerated $\it{cis}$-regulatory evolution of $\it{" + gene + "}$", size = 14)
#plt.axvline(44684478-50, color = "black")
d = np.max(zp["Pos"]) - np.min(zp["Pos"])
s1 = np.round(np.min(zp["Pos"])+d//10, -4)
s2 = np.round((np.max(zp["Pos"]) + np.min(zp["Pos"]))//2, -4)
s3 = np.round(np.max(zp["Pos"])-d//10, -4)
plt.xticks([s1, s2, s3], [chrom + ":" + str(s1), chrom + ":" + str(s2), chrom + ":" + str(s3)], size = 12)
plt.xlabel("Genomic position", size = 13)
ax.invert_xaxis()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)
gene = gene
z4 = z[z["NearestGene"] == gene]
#sns.kdeplot(data = z4, hue = "Derived", x = "PhyloP Non", common_norm = False, fill = True, palette = palette)
#plt.xlabel("Non-negative PhyloP")
#plt.title("PhyloP distribution for $\it{" + gene + "}$")
#plt.show()

z4h = z4[z4["Derived"] == "Human"]
z4c = z4[z4["Derived"] == "Chimp"]
print(mwu(z4h["PhyloP Non"].astype(float), z4c["PhyloP Non"].astype(float), alternative = "greater"))
print(fisher_exact([[z4h[z4h["PhyloP447"] >= 2].shape[0], z4c[z4c["PhyloP447"] >= 2].shape[0]], [z4h[z4h["PhyloP447"] < 2].shape[0], z4c[z4c["PhyloP447"] < 2].shape[0]]]))
print(fisher_exact([[z4h[z4h["PhyloP447"] >= 1].shape[0], z4c[z4c["PhyloP447"] >= 1].shape[0]], [z4h[z4h["PhyloP447"] < 1].shape[0], z4c[z4c["PhyloP447"] < 1].shape[0]]]))

sns.kdeplot(data = z4, hue = "Derived", x = "PhyloP Non", common_norm = False, fill = True, palette = palette)
plt.ylim(0, 0.15)
plt.xlim(1, 12)
plt.xlabel("Non-negative PhyloP")
plt.title("PhyloP distribution for $\it{" + gene + "}$")
plt.show()



In [ ]:
gene = "SIX3"
cre = pd.read_csv("CREs_To_Plot_PhyloP447.bed", sep = "\t", header = None)
cre = cre[cre[3] == "SIX3_2"].dropna()

hum_vars = list(zp[(zp["Derived"] == "Human") & (zp["KeptAfterFilt"] == "Y")]["Pos"])
chp_vars = list(zp[(zp["Derived"] == "Chimp") & (zp["KeptAfterFilt"] == "Y")]["Pos"])
cre["PhyloP Non"] = np.maximum(0, cre[7])
cre = add_moving_average(cre, output_col = "PhyloP Non MA", window_size = 5)

fig, ax = plt.subplots(figsize=(6, 4), dpi=600)
cre["Pos"] = cre[6]
plot_ma(cre)
cre2 = cre[cre[6].astype(int).isin(hum_vars + chp_vars)]

import matplotlib.transforms as transforms

PIXEL_OFFSET = 0.4  # points
MAX_LANES = 5     # limit spread

prev = 0
for i, (_, row) in enumerate(cre2.iterrows()):
    
    lane = i % MAX_LANES
    dx = (lane - (MAX_LANES - 1) / 2) * PIXEL_OFFSET

    trans = transforms.offset_copy(
        ax.transData,
        fig=fig,
        x=dx,
        y=0,
        units="points"
    )

    x = np.linspace(row["Pos"] - 0.5, row["Pos"], 10)

    if row["Pos"] in hum_vars:
        color = palette["Human"]
    elif row["Pos"] in chp_vars:
        color = palette["Chimp"]
    else:
        continue
    if row["Pos"] - prev < 5:
        print("BLEH")
        ax.fill_between(
            x,
            0,
            max(row["PhyloP Non"], 0),
            color=color,
            transform=trans,
            zorder=3
        )
    else:
        ax.fill_between(
            x,
            0,
            max(row["PhyloP Non"], 0),
            color=color,
            zorder=3
        )
    prev = row["Pos"]
plt.title("Most accelerated region near $\it{SIX3}$", size = 14)
plt.xticks([np.min(cre["Pos"]) + 50, np.max(cre["Pos"]) - 50], [chrom + ":" + str(np.min(cre["Pos"]) + 50), chrom + ":" + str(np.max(cre["Pos"]) - 50)], size = 12)
plt.xlabel("Genomic position", size = 13)
plt.ylabel("PhyloP", size = 13)
plt.yticks(size = 12)

In [ ]:
z = pd.read_csv("ITL23_AllSitesToDownload.txt.gz", sep = "\t")

In [ ]:
percent_cut = 90
cpred = z[z["Derived"] == "C"][["allele1_pred_counts", "allele2_pred_counts"]]
hpred = z[z["Derived"] == "H"][["allele1_pred_counts", "allele2_pred_counts"]]
cpred_to_cut = pd.DataFrame(np.max(cpred[["allele1_pred_counts", "allele2_pred_counts"]].astype(float), axis = 1)).sort_values(0)
hpred_to_cut = pd.DataFrame(np.max(hpred[["allele1_pred_counts", "allele2_pred_counts"]].astype(float), axis = 1)).sort_values(0)
cutoff_cpred = list(cpred_to_cut[0])[floor(percent_cut*hpred_to_cut.shape[0]//100)]
cutoff_hpred = list(hpred_to_cut[0])[floor(percent_cut*cpred_to_cut.shape[0]//100)]

cut = (cutoff_cpred + cutoff_hpred)/2
z["abs logfc"] = np.abs(z["logfc"])
zf = z[(z["allele1_pred_counts"] > cut) | (z["allele2_pred_counts"] > cut)]

In [ ]:
zf["logfc"] = -zf["logfc"]

In [ ]:
zf["Derived"] = zf["Derived"].replace("H", "Human").replace("C", "Chimp")

In [ ]:
zf["PhyloP NonNeg"] = np.maximum(0, zf["PhyloP447"])

In [ ]:
zp[zp["NearestDist"] == 0].sort_values("PhyloP447").tail(50)

In [ ]:
gene = "DPP10"
#zp = z[(z["NearestGene"] == gene) & (z["SpecSup447"] > 250)]
zp = zf[zf["NearestGene"] == gene]
#zp = zp[zp["KeptAfterFilt"] == "Y"]
#gene = 'Positive Regulation Of Type II Interferon Production (GO:0032729)'
#zp = z[(z["NearestGene"].isin(d_BP[gene])) & (z["SpecSup447"] > 250)]
#zp = zf[(zf["NearestGene"].isin(d_BP[gene]))]
sns.kdeplot(data = zp, hue = "Derived", x = "abs logfc", common_norm = False, fill = True, palette = palette)
plt.xlabel("Predicted absolute log$_{2}$ fold-change in CA")
plt.title("LFC distribution for $\it{" + gene + "}$")
plt.show()


In [ ]:
gene = "DPP10"
#zp = z[(z["NearestGene"] == gene) & (z["SpecSup447"] > 250)]
zp = zf[(zf["NearestGene"] == gene) & (zf["NearestDist"] == 0)]
#zp = zp[zp["KeptAfterFilt"] == "Y"]
#gene = 'Positive Regulation Of Type II Interferon Production (GO:0032729)'
#zp = z[(z["NearestGene"].isin(d_BP[gene])) & (z["SpecSup447"] > 250)]
#zp = zf[(zf["NearestGene"].isin(d_BP[gene]))]
sns.kdeplot(data = zp, hue = "Derived", x = "PhyloP NonNeg", common_norm = False, fill = True, palette = palette)
plt.xlabel("PhyloP")
plt.title("PhyloP distribution for $\it{" + gene + "}$")
plt.show()


In [ ]:
zph = zp[zp["Derived"] == "Human"]
zpc = zp[zp["Derived"] == "Chimp"]
print(mwu(zph["abs logfc"].astype(float), zpc["abs logfc"].astype(float), alternative = "greater"))
print(fisher_exact([[zph[zph["abs logfc"] >= 0.25].shape[0], zpc[zpc["abs logfc"] >= 0.25].shape[0]], [zph[zph["abs logfc"] < 0.25].shape[0], zpc[zpc["abs logfc"] < 0.25].shape[0]]]))
print([[zph[zph["abs logfc"] >= 0.25].shape[0], zpc[zpc["abs logfc"] >= 0.25].shape[0]], [zph[zph["abs logfc"] < 0.25].shape[0], zpc[zpc["abs logfc"] < 0.25].shape[0]]])

In [ ]:
print(fisher_exact([[zph[zph["logfc"] >= 0.25].shape[0], zph[zph["logfc"] <= -0.25].shape[0]], [zpc[zpc["logfc"] >= 0.25].shape[0], zpc[zpc["logfc"] <= -0.25].shape[0]]]))
print([[zph[zph["logfc"] >= 0.25].shape[0], zph[zph["logfc"] <= -0.25].shape[0]], [zpc[zpc["logfc"] >= 0.25].shape[0], zpc[zpc["logfc"] <= -0.25].shape[0]]])

In [ ]:
zfp = zp.set_index("Position.1")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

chrom = "chr2"
zfp["Pos"] = [int(x.split(":")[1]) for x in zfp.index]
# Sort by genomic position
zfp = zfp.sort_values("Pos")

def assign_position_clusters(df, distance=500):
    df = df.sort_values("Pos").reset_index(drop=True)

    cluster_ids = [0]
    current_cluster = 0

    for i in range(1, len(df)):
        if df.loc[i, "Pos"] - df.loc[i-1, "Pos"] <= distance:
            cluster_ids.append(current_cluster)
        else:
            current_cluster += 1
            cluster_ids.append(current_cluster)

    df = df.copy()
    df["cluster_id"] = cluster_ids
    return df

df_clustered = assign_position_clusters(zfp, distance=500)

summed = (
    df_clustered
    .groupby(["cluster_id", "Derived"], as_index=False)
    .agg(
        start=("Pos", "min"),
        end=("Pos", "max"),
        summed_logfc=("logfc", "sum"),
        n_sites=("logfc", "size")
    )
)

zfpp = summed.copy()
#zfpp["summed_logfc"] = -zfpp["summed_logfc"]

fig, ax = plt.subplots(figsize=(6, 4), dpi = 600)

zfpp["Pos"] = np.mean(zfpp[["start", "end"]], axis = 1)
zfpp = zfpp[np.abs(zfpp["summed_logfc"]) > 0.5]
zfpph = zfpp[zfpp["Derived"] == "Human"]
zfppc = zfpp[zfpp["Derived"] == "Chimp"]

ax.vlines(
    x=zfpph["Pos"],
    ymin=0,
    ymax=zfpph["summed_logfc"],
    color="gray",
    linewidth=1,
    alpha=0.7
)

ax.vlines(
    x=zfppc["Pos"],
    ymin=0,
    ymax=zfppc["summed_logfc"],
    color="gray",
    linewidth=1,
    alpha=0.7
)

# Points at the end of stems
sc = ax.scatter(
    zfpph["Pos"],
    zfpph["summed_logfc"],
    c=palette["Human"],
    cmap="RdBu_r",
    vmin=-max(abs(zfpph["summed_logfc"])),
    vmax=max(abs(zfpph["summed_logfc"])),
    s=60,
    edgecolor="black",
    zorder=3
)

sc = ax.scatter(
    zfppc["Pos"],
    zfppc["summed_logfc"],
    c=palette["Chimp"],
    vmin=-max(abs(zfppc["summed_logfc"])),
    vmax=max(abs(zfppc["summed_logfc"])),
    s=60,
    edgecolor="black",
    zorder=3
)

# Zero line
ax.axhline(0, color="black", linewidth=1)

# Labels
ax.set_xlabel("Genomic position (bp)")
ax.set_ylabel("Log2 fold-change\n(accessibility)")

# Optional: prettier x-axis formatting
ax.ticklabel_format(style="plain", axis="x")
ax.margins(x=0.01)


regions_file = pd.read_csv("All_NC_Regions.bed", sep = "\t", header = None)
regions_file = regions_file[(regions_file[3].isin(["CDS", "UTR"])) & (regions_file[4] == gene)].sort_values(2)

prev_ind = 0
prev = 0
out = []
for index, row in regions_file.iterrows():
    if prev_ind:
        if prev[2] == row[1]:
            out.append(list(row))
        else:
            out.append([row[0], prev[2], row[1], "Intron"])
            out.append(list(row))
    else:
        out.append(list(row))
        prev_ind = 1
    prev = row
region_plot = pd.DataFrame(out)

for index, row in region_plot.iterrows():
    if row[3] == "CDS":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.3*0.5, y2=-0.3*0.5, color='black')
    if row[3] == "UTR":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.13*0.5, y2=-0.13*0.5, color='black')
    if row[3] == "Intron":
        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.075*0.5, y2=-0.075*0.5, color='black')
plt.ylim(-5, 5)
#plt.xlabel("Position on " + chrom)
plt.title("$\it{" + gene + "}$ in fetal Purkinje cells", size = 14)
plt.ylabel("Summed predicted log fold-change in CA", size = 13)
d = np.max(zfp["Pos"]) - np.min(zfp["Pos"])
s1 = np.round(np.min(zfp["Pos"])+d//10, -4)
s2 = np.round((np.max(zfp["Pos"]) + np.min(zfp["Pos"]))//2, -4)
s3 = np.round(np.max(zfp["Pos"])-d//10, -4)
plt.xticks([s1, s2, s3], [chrom + ":" + str(s1), chrom + ":" + str(s2), chrom + ":" + str(s3)], size = 12)
plt.xlabel("Genomic position", size = 13)
plt.show()

In [ ]:

def add_moving_average(df, window_size, column="PhyloP Non", output_col=None):
    """
    Adds a moving average column to a presorted DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Presorted DataFrame
    window_size : int
        Size of the moving window
    column : str
        Column to compute the moving average on
    output_col : str or None
        Name of the output column (defaults to f"{column}_MA_{window_size}")

    Returns
    -------
    pandas.DataFrame
        DataFrame with moving average column added
    """
    if output_col is None:
        output_col = f"{column}_MA_{window_size}"

    df[output_col] = (
        df[column]
        .rolling(window=window_size, min_periods=1, center=True)
        .mean()
    )

    return df

def plot_ma(v, width = 2):
    x = v["Pos"]  # or df["position"] if you have one
    y = v["PhyloP Non MA"]
    
    # Choose a base color
    line_color = "grey"
    
    # Convert to RGBA and make a lighter version
    r, g, b, _ = mcolors.to_rgba(line_color)
    line_color = (r, g, b, 0.7)
    fill_color = (r, g, b, 0.2)  # lower alpha = lighter fill
        
    # Plot smoothed line
    plt.plot(x, y, color=line_color, linewidth=width, zorder = 2)
    
    # Fill between 0 and the line
    plt.fill_between(x, 0, y, color=fill_color, zorder = 1)
    
    #plt.axhline(0, color="black", linewidth=0.8)
    #plt.xlabel("Index")
    #plt.ylabel("Smoothed Moving Average")

import matplotlib.colors as mcolors

def windowed_phyloP_scores(
    zp: pd.DataFrame,
    start: int,
    end: int,
    size: int = 500,
    chrom: str | None = None,
    pos_col: str = "Pos",
    value_col: str = "PhyloP Non",
    category_col: str = "Derived",
    categories: tuple = ("Human", "Chimp"),
):
    """
    Compute windowed sums of PhyloP scores for arbitrary window sizes.

    Parameters
    ----------
    zp : pd.DataFrame
        Input dataframe containing positions, scores, and categories
    start : int
        Start coordinate (inclusive)
    end : int
        End coordinate (exclusive)
    size : int, default 500
        Window size in base pairs
    chrom : str or None
        Chromosome label to attach to output
    pos_col : str
        Column with genomic positions
    value_col : str
        Column to sum within windows
    category_col : str
        Column defining categories (e.g. Human / Chimp)
    categories : tuple
        Categories to extract and difference

    Returns
    -------
    pd.DataFrame
        Windowed summary with one row per window
    """

    df = zp.copy()

    # Ensure integer positions
    df[pos_col] = df[pos_col].astype(int)

    # Restrict to requested region
    df = df[(df[pos_col] >= start) & (df[pos_col] < end)]

    # Assign windows
    df["window_start"] = ((df[pos_col] - start) // size) * size + start
    df["window_end"] = df["window_start"] + size

    # Aggregate
    grouped = (
        df.groupby(
            ["window_start", "window_end", category_col],
            as_index=False
        )[value_col]
        .sum()
    )

    # Pivot to wide format
    wide = (
        grouped
        .pivot_table(
            index=["window_start", "window_end"],
            columns=category_col,
            values=value_col,
            fill_value=0
        )
        .reset_index()
    )

    # Ensure requested categories exist
    for cat in categories:
        if cat not in wide:
            wide[cat] = 0

    # Difference (first - second)
    if len(categories) == 2:
        wide["diff"] = wide[categories[0]] - wide[categories[1]]

    # Add chromosome column if provided
    if chrom is not None:
        wide.insert(0, "chrom", chrom)

    return wide

def make_perm(zp, perm_phylo = False, n_iters = 100, window = 3):
    zpp = zp.copy()
    for i in range(n_iters):
        if perm_phylo:
            zpp["PhyloP Non"] = list(zp.sample(frac = 1, replace = False)["PhyloP Non"])
        zpp["Derived"] = np.random.choice(a=["Human", "Chimp"], p=[0.5, 0.5], size = zpp.shape[0])
        dfp = windowed_phyloP_scores(zpp, start= np.min(zpp["Pos"]), end = np.max(zpp["Pos"]), size = 500)
        dfp["window_start"] = [x*1000 for x in range(0, dfp.shape[0])]
        dfp["window_end"] = [(x+1)*1000 for x in range(0, dfp.shape[0])]
        dfp.index = dfp["window_end"]
        dfp = dfp[["diff"]]
        dfpma = add_moving_average(dfp, window, column="diff", output_col="PhyloP Non MA " + str(i))
        dfp = dfp[["PhyloP Non MA " + str(i)]]
        
        if i:
            dfpa = dfpa.join(dfp)
        else:
            dfpa = dfp.copy()
    dfpa["Mean"] = np.mean(dfpa, axis = 1)
    dfpa["STD"] = np.std(dfpa, axis = 1)
    dfpa = dfpa[["Mean", "STD"]]
    dfpa["Lower"] = dfpa["Mean"] - dfpa["STD"]
    dfpa["Upper"] = dfpa["Mean"] + dfpa["STD"]
    return dfpa

def insert_zero_crossings(x, y):
    x = list(x)
    y = list(y)
    x_new = []
    y_new = []

    for i in range(len(y) - 1):
        x_new.append(x[i])
        y_new.append(y[i])

        # Sign change (exclude exact zeros)
        if y[i] * y[i + 1] < 0:
            # Linear interpolation for y=0
            t = -y[i] / (y[i + 1] - y[i])
            x0 = x[i] + t * (x[i + 1] - x[i])

            x_new.append(x0)
            y_new.append(0.0)

    x_new.append(x[-1])
    y_new.append(y[-1])

    return np.array(x_new), np.array(y_new)
    
def plot_ma_nc(v, width = 2, fill = True):
    
    # Choose a base color
    hline_color = palette["Human"]
    cline_color = palette["Chimp"]
    
    x = v["Pos"]  # or df["position"] if you have one
    y = v["PhyloP Non MA"].to_numpy()

    x, y = insert_zero_crossings(x, y)
    
    y_pos = np.where(y >= 0, y, np.nan)
    y_neg = np.where(y <= 0, y, np.nan)

    # Convert to RGBA and make a lighter version
    r, g, b, _ = mcolors.to_rgba(hline_color)
    hline_color = (r, g, b, 1)
    hfill_color = (r, g, b, 0.5)  # lower alpha = lighter fill

    r, g, b, _ = mcolors.to_rgba(cline_color)
    cline_color = (r, g, b, 1)
    cfill_color = (r, g, b, 0.5)  # lower alpha = lighter fill
    
    plt.plot(x, y_pos, color=hline_color, linewidth=2)
    plt.plot(x, y_neg, color=cline_color, linewidth=2)

    if fill:
        # Fill between 0 and the line
        plt.fill_between(x, 0, y, color=hfill_color, where=(y > 0), zorder = 1)
        plt.fill_between(x, 0, y, color=cfill_color, where=(y < 0), zorder = 1)
    
    #plt.axhline(0, color="black", linewidth=0.8)
    #plt.xlabel("Index")
    #plt.ylabel("Smoothed Moving Average")


In [ ]:
size = 500
zfp["Pos"] = [int(x.split(":")[1]) for x in zfp.index]
# Sort by genomic position
zfp = zfp.sort_values("Pos")
#zfp["Pos"] = list(range(1, zfp.shape[0] + 1))
zfp["PhyloP Non"] = zfp["abs logfc"]
chrom = list(zfp.index)[0].split(":")[0]
df = windowed_phyloP_scores(zfp, start= np.min(zfp["Pos"]), end = np.max(zfp["Pos"]), size = size)
dfc = df.copy()
df["window_start"] = [x*1000 for x in range(0, df.shape[0])]
df["window_end"] = [(x+1)*1000 for x in range(0, df.shape[0])]
df

In [ ]:
c = 0
ma_window_size = 20
plt.figure(figsize=(6, 4), dpi=600)
dfpa = make_perm(zfp, window = ma_window_size)

r, g, b, _ = mcolors.to_rgba("grey")
fill_color = (r, g, b, 0.2)
plt.plot(list(dfpa.index), dfpa["Lower"], color="black", zorder = 3, linewidth = 0.8)
plt.plot(list(dfpa.index), dfpa["Upper"], color="black", zorder = 3, linewidth = 0.8)

dfma = add_moving_average(df, ma_window_size, column="diff", output_col="PhyloP Non MA")
dfma["Pos"] = dfma["window_end"]
plot_ma_nc(dfma, width = 0.5, fill = True)

#regions_file = pd.read_csv("All_NC_Regions.bed", sep = "\t", header = None)
#regions_file = regions_file[(regions_file[3].isin(["CDS", "UTR"])) & (regions_file[4] == gene)].sort_values(2)

#prev_ind = 0
#prev = 0
#out = []
#for index, row in regions_file.iterrows():
#    if prev_ind:
#        if prev[2] == row[1]:
#            out.append(list(row))
#        else:
#            out.append([row[0], prev[2], row[1], "Intron"])
#            out.append(list(row))
#    else:
#        out.append(list(row))
#        prev_ind = 1
#    prev = row
#region_plot = pd.DataFrame(out)

#adjust = (np.max(dfma["PhyloP Non MA"]) - np.min(dfma["PhyloP Non MA"]))/10
#for index, row in region_plot.iterrows():
#    if row[3] == "CDS":
#        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.3*adjust, y2=-0.3*adjust, color='black', zorder = 4)
#    if row[3] == "UTR":
#        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.13*adjust, y2=-0.13*adjust, color='black', zorder = 4)
#    if row[3] == "Intron":
#        plt.fill_between(np.linspace(row[1], row[2], row[2] - row[1]), y1=0.075*adjust/2, y2=-0.075*adjust/2, color='black', zorder = 4)
plt.ylabel("Difference in predicted absolute\nlog fold-change in CA moving average", size = 13)
plt.title("$\it{" + gene + "}$ in fetal Purkinje cells", size = 14)
#d = np.max(zfp["Pos"]) - np.min(zfp["Pos"])
s1 = np.round(np.min(dfma["window_start"]), -4)
s1l = np.round(np.min(zfp["Pos"]), -4)
#s2 = np.round((np.max(zfp["Pos"]) + np.min(zfp["Pos"]))//2, -4)
s3 = np.round(np.max(dfma["window_end"]), -4)
s3l = np.round(np.max(zfp["Pos"]), -4)

plt.xticks([s1, s3], [chrom + ":" + str(s1l), chrom + ":" + str(s3l)], size = 12)
plt.ylim(-0.3, 0.3)
#plt.xlabel("Genomic position", size = 13)
#plt.xlabel("")
#plt.xticks([], [])
#plt.ylabel("")
#plt.yticks([], [])
#plt.title("")
plt.show()